# Experiment: Case 001 Transfusion Burden Quantification Gate

Objective:
- Keep weekly transfusion as `transfusion_dependent_burden_unquantified` until the minimum quantitative fields are present.
- Verify a synthetic, de-identified calculation path for interval, `ml/kg/year`, pure red-cell volume, and iron input.
- Keep all patient-specific interpretation blocked for clinician review.


In [ ]:
from __future__ import annotations

import datetime as dt
import importlib
import sys
from dataclasses import dataclass
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

burden = importlib.import_module("calc_transfusion_burden")

## Synthetic Calculation Check

This uses synthetic rows only. It checks that the calculator can convert a weekly-like log into annual volume, pure red-cell volume, and estimated iron input without storing private records.


In [ ]:
synthetic_rows = [
    burden.TransfusionRow(
        date=dt.date(2026, 1, day),
        body_weight_kg=50.0,
        transfused_volume_ml=250.0,
        red_cell_fraction=0.65,
        pre_hb_g_dl=8.3,
        post_hb_g_dl=10.3,
    )
    for day in (1, 8, 15, 22, 29)
]

summary = burden.summarize_rows(synthetic_rows)
selected_summary = {
    "mean_interval_days": summary["mean_interval_days"],
    "annual_volume_ml_per_kg": summary["annual_volume_ml_per_kg"],
    "annual_pure_red_cell_ml_per_kg": summary["annual_pure_red_cell_ml_per_kg"],
    "daily_iron_loading_mg_per_kg": summary["daily_iron_loading_mg_per_kg"],
    "warnings": summary["warnings"],
}
selected_summary

## Missing-Record Gate

The current public case label should remain `transfusion_dependent_burden_unquantified` until the high-value fields below are known or explicitly marked unavailable by clinicians.


In [ ]:
@dataclass(frozen=True)
class RecordField:
    """A missing record field and the analyses it blocks."""

    name: str
    status: str
    blocks_interval: bool
    blocks_volume: bool
    blocks_iron_input: bool
    blocks_response: bool
    request: str

    def score(self) -> int:
        """Count the analysis lanes blocked by the missing field."""
        flags = [
            self.blocks_interval,
            self.blocks_volume,
            self.blocks_iron_input,
            self.blocks_response,
        ]
        return sum(1 for flag in flags if flag)


fields = [
    RecordField(
        name="transfusion_dates",
        status="missing",
        blocks_interval=True,
        blocks_volume=True,
        blocks_iron_input=True,
        blocks_response=False,
        request="Ask for the dated transfusion log covering 6-12 months.",
    ),
    RecordField(
        name="body_weight_at_visits",
        status="missing",
        blocks_interval=False,
        blocks_volume=True,
        blocks_iron_input=True,
        blocks_response=False,
        request="Ask for weight paired with each transfusion visit.",
    ),
    RecordField(
        name="transfused_volume_or_units",
        status="missing",
        blocks_interval=False,
        blocks_volume=True,
        blocks_iron_input=True,
        blocks_response=True,
        request="Ask for ml volume or unit count for each visit.",
    ),
    RecordField(
        name="red_cell_fraction_or_unit_hematocrit",
        status="missing",
        blocks_interval=False,
        blocks_volume=False,
        blocks_iron_input=True,
        blocks_response=False,
        request="Ask the blood bank for product hematocrit or Hb content.",
    ),
    RecordField(
        name="pre_and_post_transfusion_hb",
        status="missing",
        blocks_interval=False,
        blocks_volume=False,
        blocks_iron_input=False,
        blocks_response=True,
        request="Ask for pre-Hb and post-Hb or Hb increment if available.",
    ),
]

ranked_missing = sorted(fields, key=lambda field: field.score(), reverse=True)
ranked_output = [(field.name, field.score(), field.request) for field in ranked_missing]
ranked_output

## Result

The label remains conservative. The notebook only prepares a record request and a synthetic calculation check; it does not infer whether the real schedule is appropriate.


In [ ]:
minimum_packet = [field.name for field in ranked_missing if field.score() >= 2]
result = {
    "case_label": "transfusion_dependent_burden_unquantified",
    "minimum_packet": minimum_packet,
    "synthetic_interval_days": summary["mean_interval_days"],
    "synthetic_daily_iron_mg_per_kg": summary["daily_iron_loading_mg_per_kg"],
    "decision": "keep patient relevance blocked pending clinician review",
}

assert result["case_label"] == "transfusion_dependent_burden_unquantified"
assert "transfusion_dates" in result["minimum_packet"]
assert "transfused_volume_or_units" in result["minimum_packet"]
assert summary["warnings"] == []
result